<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [18]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed):
    data = fetch_openml('Fashion-MNIST', version=1, as_frame=False)
    
    X = data.data
    y = data.target.astype(int)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    return X_train, X_test, y_train, y_test

Não é necessário normalizar os dados para Random Forest e AdaBoost.

Esses modelos são baseados em árvores de decisão, que não dependem da escala das variáveis. Portanto, a magnitude dos dados não impacta diretamente o desempenho.

Ainda assim, a normalização pode ser utilizada em pipelines por padronização do processo, especialmente ao comparar com outros modelos.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [19]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed):
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed
    )
    model.fit(X_train, y_train)
    return model


def train_adaboost(X_train, y_train, seed):
    model = AdaBoostClassifier(
        n_estimators=50,
        random_state=seed
    )
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [20]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    return acc

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [21]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)
    
    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")
    
    acc = evaluate(model, X_test, y_test)
    
    return acc

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

O overfitting começa a ocorrer quando o desempenho no conjunto de treino continua aumentando enquanto o desempenho no conjunto de teste começa a diminuir, indicando perda de capacidade de generalização. Não há uma profundidade fixa para isso, pois depende do dataset.

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

O Random Forest apresentou melhor desempenho inicial em comparação ao AdaBoost, obtendo valores superiores nas métricas de acurácia, precisão, recall e F1-score.

Isso ocorre porque o Random Forest combina múltiplas árvores de decisão, reduzindo variância e melhorando a generalização. Já o AdaBoost é mais sensível a ruídos e erros nas amostras, o que pode impactar seu desempenho inicial.

**Solução**:

In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_full(model, X_test, y_test):
    y_pred = model.predict(X_test)
    
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted"),
        "recall": recall_score(y_test, y_pred, average="weighted"),
        "f1": f1_score(y_test, y_pred, average="weighted")
    }

In [23]:
seed = 42

# carregar dados
X_train, X_test, y_train, y_test = load_data(seed)

# treinar modelos
rf = train_random_forest(X_train, y_train, seed)
ab = train_adaboost(X_train, y_train, seed)

# avaliar
rf_metrics = evaluate_full(rf, X_test, y_test)
ab_metrics = evaluate_full(ab, X_test, y_test)

print("Random Forest:", rf_metrics)
print("AdaBoost:", ab_metrics)

Random Forest: {'accuracy': 0.8818571428571429, 'precision': 0.8807269240971383, 'recall': 0.8818571428571429, 'f1': 0.8800202613715961}
AdaBoost: {'accuracy': 0.49092857142857144, 'precision': 0.4717382248567284, 'recall': 0.49092857142857144, 'f1': 0.44483625605040067}


# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

Os resultados apresentaram pequenas variações ao utilizar seeds diferentes, o que é esperado devido à aleatoriedade no processo de divisão dos dados e no treinamento dos modelos.

## Responda:
- O experimento é reprodutível? Justifique.

Apesar dessas variações, o experimento é reprodutível, pois ao utilizar a mesma seed, os resultados obtidos são sempre os mesmos. O uso do parâmetro random_state garante que todas as etapas que envolvem aleatoriedade sejam controladas, permitindo replicar os resultados de forma consistente.

**Solução**:

In [28]:
seeds = [42, 7]

for seed in seeds:
    rf_acc = run_pipeline("rf", seed)
    ab_acc = run_pipeline("ab", seed)
    
    print(f"Seed {seed}")
    print(f"Random Forest: {rf_acc}")
    print(f"AdaBoost: {ab_acc}")
    print("-" * 30)

Seed 42
Random Forest: 0.8818571428571429
AdaBoost: 0.49092857142857144
------------------------------
Seed 7
Random Forest: 0.8800714285714286
AdaBoost: 0.45535714285714285
------------------------------


# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [25]:
# TODO: implemente load_data

# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [26]:
# TODO: implemente load_data

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

In [27]:
# TODO: implemente load_data